# 1-Generate Predictions using LangChain

- **Goal:** Use LLMs to generate textual predictions for the following domains: financial, health, policy, weather, sports, and miscellaneous. 

- **Code Structure:** 

    1. Base template: This is included in every domain's input.
    2. Domain template: Vary or specific to a domain.

- **Run Notebook:**

    1. See README.md for installation and initial setup.
    2. Choose models (from `text_generation_models.py`) to generate data.
    3. Here in JupyerNotebook, click `Run All` button at top/in menu bar.
    4. Reach out if any problems.

In [1]:
import os
import sys

import pandas as pd

from datetime import datetime

from langchain_core.prompts import PipelinePromptTemplate, PromptTemplate
# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

from log_files import LogData
from data_processing import DataProcessing
from text_generation_models import TextGenerationModelFactory

## Base Template for Domain Predictions

- `{prediction_properties}:` These are the variables for each prediction.
- `{prediction_requirements}:` These are to state how outcome predictions should be limited to or expressed as.
- `{prediction_templates}:` These are to give LLMs proper structure/syntax.
- `{prediction_examples}:` These are to provide LLMs with examples that match the templates.

In [2]:
full_prediction_template = """{prediction_properties}

{prediction_requirements}

{prediction_templates}

{prediction_examples}
"""
full_prediction_prompt = PromptTemplate.from_template(full_prediction_template)

In [ ]:
prediction_properties_template = """A prediction <p> = (<p_s>, <p_t>, <p_d>, <p_a>, <p_m>, and <p_sl>), where it consists of the following six properties:

    1. <p_s>, any source entity in the {prediction_domain} domain.
        - Can be a person (with a name) or a {prediction_domain} person such as a {prediction_domain} reporter, {prediction_domain} analyst, {prediction_domain} expert, {prediction_domain} top executive, {prediction_domain} senior level person, etc), civilian.
        - Can only be an organization that is associated with the {prediction_domain} prediction.
    2. <p_t>, any target entity in the {prediction_domain} domain.
	    - Can be a person (with a name) or a {prediction_domain} person such as a {prediction_domain} reporter, {prediction_domain} analyst, {prediction_domain} expert, {prediction_domain} top executive, {prediction_domain} senior level person, etc).
        - Can only be an organization that is associated with the {prediction_domain} prediction.
    3. <p_d>, date or time range when <p> is expected to come to fruition or when one should observe the <p>.
        - Forecast can range from a second to anytime in the future.
        - Answers the questions: "How far to go out from today?" or "Where to stop?".
    4. <p_a>, {prediction_domain} prediction outcome atribute.
        - Identifies what aspect or characteristic of the target entity the prediction is about
        - Some examples are {prediction_domain_outcome}.
    5. <p_m>, a quantifiable metric used to measure <p_a> from the {prediction_domain} domain.
        - Specifies how the attribute will be measured or expressed (e.g., a percentage, a price, a count, a ratio, a score).
        - Must be concrete and observable so that the prediction can be verified.
        - Some examples are: 15%, 3% to 10%, 3.2 percent.
    6. <p_sl>, the expected direction of change in <p_m> from the {prediction_domain} domain.
        - Describes *which way* the metric is expected to move: increasing, decreasing, or staying stable.
        - Expressed qualitatively (e.g., "rise", "fall", "remain flat", "exceed", "drop below", "spike", "dip").
        - The magnitude of the change is captured by <p_m>, not here.
"""

prediction_properties_prompt = PromptTemplate.from_template(prediction_properties_template)

In [4]:
prediction_requirements_template = """{prediction_domain} requirements to use for each prediction:

    - Should be based on real-world {prediction_domain} data and not hallucinate.
    - Only a simple sentence (prediction) (and NOT compounding using "and" or "or").
    - Should diversify all four properties of the prediction (<p>) as in change and not use same for <p_s>, <p_t>, <p_d>, <p_a>, <p_m>, <p_sl>.
    - Should use synonyms to predict such as forecasts, speculates, foresee, envision, etc., and not use any of them more than ten times.
    - The prediction should be unique and not repeated.
    - Do not number the predictions.
    - Do not say, "As the {prediction_domain}, I will generate company-based {prediction_domain} predictions using the provided templates." or anything similar.
    - Use the five different templates and examples provided.
    - Change how the current date (<p_d>) written in the prediction with examples of (1) Wednesday, August 21, 2024; (2) Wed, August 21, 2024; (3) 08/21/2024; (4) 08/21/2024; (5) 21/08/2024; (6) 21 August 2024; (7) 2024/08/21; (8) 2024-08-21; (9) August 21, 2024; (10) Aug 21, 2024; (11) 21 August 2024, (12) 21 Aug 2024, Q3 of 2027, 2029 of Q3, etc (with removing day of week).
    {domain_requirements}
    - Do not say, "Here are {predictions_N} unique {prediction_domain} predictions based on the provided templates and examples:" in the prompt.
    - Do not use any of the examples in the prompt.
    - In front of every prodiction, put the template number in the format of "T1:", "T2:", etc. and do not number them like "1.", "2.", etc. Should have template number and generated prediction matching.
    - Do not put template number on line by itself. Always pair with a prediction.
    - Disregard brackets: "<>"
    - Should never say "Here are {predictions_N} unique {prediction_domain} predictions based on the provided templates and examples:" or "Note: I've made sure to follow the guidelines and templates provided, and generated unique predictions that meet the requirements."
    - Do not use person name of entity name more than once as in don't use name Joe as both the <p_s> and <p_t>, unless like Mr. Sach and Goldman Sach or Mr. Sam Walton and Sam's Club, etc.
    - The source entity (<p_s>) is rarely the same as the target entity (<p_t>) and if same, the <p_s> is making a prediction on itself in the <p_t>.
    - Should variate the slope of rise/increase/as much as, fall/decrease/as little as, change, stay stable, high/low chance/probability/degree of, etc.
    - Should variate the prediction verbs such as will, would, be going to, should, etc.
    - Be sure to NOT use same properties in examples.
    - You must differ each property per template.
"""
prediction_requirements_prompt = PromptTemplate.from_template(prediction_requirements_template)

In [5]:
# Mapping of each template to the distinct trend‑type keywords that appear in its examples

trend_type_dict = {
    "template 1": [
        "statement value",
        "volatility peaked",
        "changes over interval",
        "second‑order effect accelerated",
        "comparison higher",
    ],
    "template 2": [
        "average",
        "volatility peaked",
        "statistical function minimum",
        "second‑order effect accelerated",
    ],
    "template 3": [
        "average value",
        "volatility peaked",
        "statistical function average",
        "second‑order effect accelerated",
        "recurrent pattern cycle",
        "comparison higher",
    ],
    "template 4": [
        "statistical function minimum",
        "volatility peaked",
        "average",
    ],
    "template 5": [
        "recurrent pattern cycle",
        "statistical function average",
    ],
    "template 6": [
        "recurrent pattern cycle",
        "statistical function average",
    ],
}

In [6]:
prediction_template_1 = "template 1: <p_s> forecasts that the <p_a> at <p_t> potentially <p_sl> in <p_d>."
prediction_template_2 = "template 2: On <p_d>, <p_s> speculates the <p_a> at <p_t> will likely <p_sl> by <p_m>%."

prediction_templates = [prediction_template_1, prediction_template_2]

In [7]:
prediction_templates_template = """Here are some {prediction_domain} templates:

    - {prediction_domain} template 1: <p_s> forecasts that the <p_a> at <p_t> potentially <p_sl> in <p_d>.
    - {prediction_domain} template 2: On <p_d>, <p_s> speculates the <p_a> at <p_t> will likely <p_sl> <p_m>.

"""
prediction_templates_prompt = PromptTemplate.from_template(prediction_templates_template)

In [8]:
generate_predictions_first = False

if generate_predictions_first: 
    prediction_examples_template = """Here are some examples of {prediction_domain} predictions:
    {domain_examples}

    With the above (prediction with four properties, requirements, templates, and examples), 
    1. generate a unique set of {predictions_N} predictions per template following the examples. Think from the perspective of an {prediction_domain} analyst, expert, top executive, or senior level person and even a college student, professional, research advisor, etc.
    2. from the prediction sentence you generate, clearly state the <p_s>, <p_t>, <p_d>, <p_a>, <p_m>, and <p_sl>. Don't hallucinate and don't use another prediction sentence, only this current prediction sentence."""

else:
    prediction_examples_template = """Here are some examples of {prediction_domain} predictions:
    {domain_examples}

    With the above (prediction with properties, requirements, templates, and examples), per template, 
    1. generate each property: <p_s>, <p_t>, <p_d>, <p_a>, <p_m>, and <p_sl> and be sure to NOT use same properties in examples and differ each property per template then
    2. generate a unique set of {predictions_N} predictions per template following the examples and using the generated properties. Think from the perspective of an {prediction_domain} analyst, expert, top executive, or senior level person and even a college student, professional, research advisor, etc.
    
    """

prediction_examples_prompt = PromptTemplate.from_template(prediction_examples_template)

# Also, state the trend types

In [9]:
prediction_input_prompts = [
    ("prediction_properties", prediction_properties_prompt),
    ("prediction_requirements", prediction_requirements_prompt),
    ("prediction_templates", prediction_templates_prompt),
    ("prediction_examples", prediction_examples_prompt),
]
pipeline_prompt = PipelinePromptTemplate(
    final_prompt=full_prediction_prompt, pipeline_prompts=prediction_input_prompts
)

C:\Users\adria\AppData\Local\Temp\ipykernel_21880\2065471046.py:7: LangChainDeprecationWarning: This class is deprecated in favor of chaining individual prompts together.
  pipeline_prompt = PipelinePromptTemplate(


## Specific Templates for Domain Predictions

- For now, generating 1 prediction per template. From here, I'll try 3 and increase by increments/multiples of 3.

- With 1 prediction per template,
    - 1 prediction per template x 6 examples per domain so 6 predictions per domain
    - 6 predictions per domain x 4 domains = 24 predictions per model
    - 24 predictions per model x 2 models = 48 predictions across all models
    - 48 predictions across all models x 2 batches = 96 across all batches

In [10]:
examples_per_template = 1
generate_N_predictions_per_template = 1 * examples_per_template

### Template for Financial Predictions

In [11]:
financial_outcomes = """stock price, net profit, revenue, operating cash flow, research and development expenses, operating income, gross profit."""
financial_requirements = """- Should be based on real-world financial earnings reports.
   - Suppose the time when $p$ was made is during any earning season.
   - Include stocks from all sectors such as consumer staples, energy, finance, health care, industrials, materials, media, real estate, retail, technology, utilities, defense, etc.
   - Include the US Dollar sign ($) before or USD after the amount of the financial outcome."""

financial_examples = """
   - financial examples for template 1:
      - <p_s>: Senator Emily Johnson, an investor
      - <p_a>: stock price
      - <p_t>: Coca-Cola
      - <p_sl>: decrease
      - <p_d>: 2027 of Q3
      - T1: Senator Emily Johnson, an investor forsees that the stock price at Coca-Cola will decrease in 2027 of Q3.
   - financial examples for template 2:
      - <p_d>: April 2, 2000
      - <p_s>: a reprentative at Morgan Stanley
      - <p_a>: stock value
      - <p_t>: Tesla
      - <p_sl>: fall
      - <p_m>: by 15%
      - T2: On April 2, 2000, a reprentative at Morgan Stanley forecasts that the stock value at Tesla should fall by 15%.
"""
financial_input_dict = {
    "prediction_domain": "financial",
    "prediction_domain_outcome": financial_outcomes,
    "domain_requirements": financial_requirements,
    "domain_examples": financial_examples,
    "predictions_N": generate_N_predictions_per_template
}
financial_prompt_output = pipeline_prompt.format(**financial_input_dict)
print(financial_prompt_output)


A prediction <p> = (<p_s>, <p_t>, <p_d>, <p_a>, <p_m>, and <p_sl>), where it consists of the following seven properties:

    1. <p_s>, any source entity in the financial domain.
        - Can be a person (with a name) or a financial person such as a financial reporter, financial analyst, financial expert, financial top executive, financial senior level person, etc), civilian.
        - Can only be an organization that is associated with the financial prediction.
    2. <p_t>, any target entity in the financial domain.
	    - Can be a person (with a name) or a financial person such as a financial reporter, financial analyst, financial expert, financial top executive, financial senior level person, etc).
        - Can only be an organization that is associated with the financial prediction.
    3. <p_d>, date or time range when <p> is expected to come to fruition or when one should observe the <p>.
        - Forecast can range from a second to anytime in the future.
        - Answers th

###  Template for Health Predictions

In [12]:
health_outcomes = """obesity rates, prevalence of chronic illnesses, average physical activity levels, nutritional intake, etc."""
health_requirements = """- Should be based on real-world health reports.
    - Suppose the time when $p$ was made is during any season such as flu season, allergy season, pandemic, epidemic, etc.
    - Include reports from all Health organization, researcher, doctor, physical therapist, physician assistant, nurse practictioners, fitness expert, etc."""
health_examples = """
    - health examples for template 1:
        - <p_s>: CDC
        - <p_a>: obesity rates
        - <p_t>: urban health centers in Dallas, TX
        - <p_sl>: dip
        - <p_d>: late 2025
        - T1: CDC forecasts that the obesity rates at urban health centers in Dallas, TX will drastically dip by late 2025.
    - health examples for template 2:
        - <p_d>: 1/2/2015
        - <p_s>: Joseph (a representative at UF Shands Hospital)
        - <p_a>: physical activity levels
        - <p_t>: U.S. high schools
        - <p_sl>: spike
        - <p_m>: from 7% to 70%.
        - T2: On 1/2/2015, Joseph (a representative at UF Shands Hospital) speculates that the physical activity levels at U.S. high schools could spike from 7% to 70%.
"""
health_input_dict = {
    "prediction_domain": "health",
    "prediction_domain_outcome": health_outcomes,
    "domain_requirements": health_requirements,
    "domain_examples": health_examples,
    "predictions_N": generate_N_predictions_per_template
}
health_prompt_output = pipeline_prompt.format(**health_input_dict)

###  Template for Policy Predictions

In [13]:
policy_outcomes = """election outcomes, economic reforms, legislative impacts."""
policy_requirements = """- Should be based on real-world policy reports.
   - Suppose the time when $p$ was made is during an election cycle or non-election cycles.
   - Include policies & laws, from all sectors such as consumer staples, energy, finance, health care, industrials, materials, media, real estate, retail, technology, utilities, defense, etc."""

policy_examples = """
   - policy examples for template 1:
        - <p_s>: analyst Rachel Lin
        - <p_a>: Election outcomes
        - <p_t>: key swing states
        - <p_sl>: rise
        - <p_d>: 1st of November 2011
        - T1: Election outcomes in key swing states should rise in national importance in the 1st of November 2011, according to analyst Rachel Lin.
   - policy examples for template 2:
        - <p_d>: 4/5/2029
        - <p_s>: International Monetary Fund
        - <p_a>: investment activity
        - <p_t>: emerging markets
        - <p_sl>: decrease
        - <p_m>: by 5%
        - T2: On May 12, 2020, the International Monetary Fund speculates that investment activity at emerging markets will decrease by 5%.
"""
policy_input_dict = {
    "prediction_domain": "policy",
    "prediction_domain_outcome": policy_outcomes,
    "domain_requirements": policy_requirements,
    "domain_examples": policy_examples,
    "predictions_N": generate_N_predictions_per_template
}
policy_prompt_output = pipeline_prompt.format(**policy_input_dict)

###  Template for Weather Predictions

In [14]:
weather_outcomes = """temperature, precipitation, wind speed, humidity, etc."""
weather_requirements = """- Should be based on real-world weather reports.
    - Suppose the time when $p$ was made is during any season and any location (ie: Florida known for hurricanes, California known for wildfires, etc).
    - Include reports from all meteorologists, weather organizations, or any type of weather entity."""
weather_examples = """
    - weather examples for template 1:
        - <p_s>: Sam Arnold a weather expert
        - <p_a>: precipitation levels
        - <p_t>: Miami
        - <p_sl>: stay stable
        - <p_d>: early fall 2025
        - T1: Sam a weather expert forecasts that the precipitation levels at Miami may stay stable in early fall 2025.
    - weather examples for template 2:
        - <p_d>: 2025 of June
        - <p_s>: San Francisco's meteorological team
        - <p_a>: wind speed
        - <p_t>: Los Angeles
        - <p_sl>: increase
        - <p_m>: from 5% to 20%
        - T2: On 2025 of June, San Francisco's meteorological team speculates that the wind speed in Los Angeles potentially could increase from 5% to 20%.
"""
weather_input_dict = {
    "prediction_domain": "weather",
    "prediction_domain_outcome": weather_outcomes,
    "domain_requirements": weather_requirements,
    "domain_examples": weather_examples,
    "predictions_N": generate_N_predictions_per_template
}
weather_prompt_output = pipeline_prompt.format(**weather_input_dict)

### Template for Sports Predictions

In [15]:
sport_outcomes = """score, touchdown, goal, points, win, lose, etc."""
sport_requirements = """- Should be based on real-world sports.
    - Suppose the time when $p$ was made is during any season of sports.
    - Include reports from all sports professionals, coaches, or any type of sport entity."""
sport_examples = """
    - sport examples for template 1:
        - <p_s>: Coach Lisa Martinez
        - <p_a>: goal average
        - <p_t>: Manchester United
        - <p_sl>: stay the same
        - <p_d>: November 2025
        - T1: Coach Lisa Martinez predicts that the goal average at Manchester United will stay the same in November 2025.
    - sport examples for template 2:
        - <p_d>: Sep 20, 2100
        - <p_s>: Analyst David Kim
        - <p_a>: touchdown rate
        - <p_t>: Kansas City Chiefs
        - <p_sl>: surge
        - <p_m>: by 10%
        - T2: On Sep 20, 2100, Analyst David Kim anticipates the touchdown rate at the Kansas City Chiefs will likely surge by 10%.
"""
sport_input_dict = {
    "prediction_domain": "sport",
    "prediction_domain_outcome": sport_outcomes,
    "domain_requirements": sport_requirements,
    "domain_examples": sport_examples,
    "predictions_N": generate_N_predictions_per_template
}
sport_prompt_output = pipeline_prompt.format(**sport_input_dict)

### Template for Miscellaneous Predictions

In [16]:
miscellaneous_outcomes = """These outcomes will take in any random outcome relating ot any real world situation."""
miscellaneous_requirements = """These outcomes will take in any random outcome relating ot any real world situation.
    - Suppose the time when $p$ was made is during any season or part of the year.
    - Include any type of entity."""
miscellaneous_examples = """
    - miscellaneous examples for template 1:
        - <p_s>: Chef Alberto Reyes
        - <p_a>: sweetness level
        - <p_t>: bakery’s pastries
        - <p_sl>: decline
        - <p_d>: fall 2027
        - T1: Chef Alberto Reyes predicts that the sweetness level in the bakery’s pastries will decline in fall 2027.
    - miscellaneous examples for template 2:
        - <p_d>: November 2, 2020
        - <p_s>: Professor Maria Jackson
        - <p_a>: GPA
        - <p_t>: Eastwood College
        - <p_sl>: go up
        - <p_m>: from 5% to 15%
        - T2: On November 2, 2020, Professor Maria Jackson speculates that the GPA at Eastwood College will go up from 5% to 15%.
"""
miscellaneous_input_dict = {
    "prediction_domain": "miscellaneous",
    "prediction_domain_outcome": miscellaneous_outcomes,
    "domain_requirements": miscellaneous_requirements,
    "domain_examples": miscellaneous_examples,
    "predictions_N": generate_N_predictions_per_template
}
miscellaneous_prompt_output = pipeline_prompt.format(**miscellaneous_input_dict)

## Generate Predictions

### Text Generation Models

In [17]:
tgmf = TextGenerationModelFactory(temperature=0.8, top_p=0.8)

# Groq Cloud (https://console.groq.com/docs/overview)
# llama_31_8b = tgmf.create_instance('llama-3.1-8b-instant')
# llama_318b_instant_generation_model = tgmf.create_instance('llama-3.1-8b-instant') 
# llama_3370b_versatile_generation_model = tgmf.create_instance('codestral-22b')
llama_3370b_versatile_generation_model = tgmf.create_instance('llama-3.3-70b-versatile')
# llama_guard_4_12b_generation_model = tgmf.create_instance('meta-llama/llama-guard-4-12b')
text_generation_models_groqcloud = [llama_3370b_versatile_generation_model]

In [18]:
today = datetime.today()
date = today.strftime("%Y-%B-%d")
print(date)

2026-March-21


In [19]:
N_batches = 1
prediction_domains = ["finance", "health", "policy", "weather", "sport", "miscellaneous"]
prediction_prompt_outputs = {
     "finance": financial_prompt_output,
     "health": health_prompt_output,
     "policy": policy_prompt_output,
     "weather": weather_prompt_output,
     "sport": sport_prompt_output,
     "miscellaneous": miscellaneous_prompt_output,
 }

prediction_label = 1
save_logs = "data"
batched_predictions_df = tgmf.batch_generate_data(N_batches=N_batches, 
                                text_generation_models=text_generation_models_groqcloud, 
                                domains=prediction_domains,
                                prompt_outputs=prediction_prompt_outputs,
                                sentence_label=prediction_label,
                                save_path=save_logs, 
                                batch_prediction_date=date, 
                                prediction_templates=prediction_templates)

  0%|          | 0/1 [00:00<?, ?it/s]

===================================== Batch 0 ===============================================
finance --- llama-3.3-70b-versatile --- GROQ_CLOUD
generates:
{"p_s": "Goldman Sachs", "p_t": "Microsoft", "p_d": "2025-08-20", "p_a": "operating income", "p_m": "", "p_sl": "increase", "Base Sentence": "T1: Goldman Sachs forecasts that the operating income at Microsoft potentially increase in 2025-08-20"}
{"p_s": "JPMorgan Chase", "p_t": "Amazon", "p_d": "August 15, 2024", "p_a": "revenue", "p_m": "by 20%", "p_sl": "rise", "Base Sentence": "T2: On August 15, 2024, JPMorgan Chase speculates the revenue at Amazon will likely rise by 20%"}

health --- llama-3.3-70b-versatile --- GROQ_CLOUD
generates:
{"p_s": "American Heart Association", "p_t": "rural communities", "p_d": "August 15, 2027", "p_a": "heart disease prevalence", "p_m": "", "p_sl": "decrease", "Base Sentence": "T1: American Heart Association forecasts that the heart disease prevalence at rural communities potentially decrease in Augu

100%|██████████| 1/1 [00:04<00:00,  4.65s/it]

generates:
{"p_s": "Dr. Sophia Patel", "p_t": "GreenTech Inc.", "p_d": "Q2 2025", "p_a": "revenue growth", "p_m": "", "p_sl": "increase", "Base Sentence": "T1: Dr. Sophia Patel forecasts that the revenue growth at GreenTech Inc. potentially increase in Q2 2025."}
{"p_s": "Financial Analyst David Lee", "p_t": "Wall Street", "p_d": "2024-08-15", "p_a": "stock prices", "p_m": "by 10% to 20%", "p_sl": "rise", "Base Sentence": "T2: On 2024-08-15, Financial Analyst David Lee speculates the stock prices at Wall Street will likely rise by 10% to 20%."}

Start logging batch
log_directory: C:\Users\adria\OneDrive\Área de Trabalho\UF Data Studio\predictions\data/prediction_logs
Save CSV: C:\Users\adria\OneDrive\Área de Trabalho\UF Data Studio\predictions\data/prediction_logs\batch_82-prediction\batch_82-from_df.csv

CSV to Log


In [20]:
pd.set_option('max_colwidth', 800)
pd.set_option('display.max_columns', 800)
pd.set_option('display.max_rows', 800)
batched_predictions_df

,Base Sentence,Source,Target,Prediction Date,Attribute,Metric,Slope,Sentence Label,Domain,Metadata
0,Goldman Sachs forecasts that the operating income at Microsoft potentially increase in 2025-08-20,Goldman Sachs,Microsoft,2025-08-20,operating income,,increase,1,finance,"{'model_name': 'llama-3.3-70b-versatile', 'api_name': 'GROQ_CLOUD', 'batch_id': 0, 'temperature': 0.8, 'top_p': 0.8, 'generation_date': '2026-March-21', 'template_number': 1, 'template_text': 'template 1: <p_s> forecasts that the <p_a> at <p_t> potentially <p_sl> in <p_d>.'}"
1,"On August 15, 2024, JPMorgan Chase speculates the revenue at Amazon will likely rise by 20%",JPMorgan Chase,Amazon,"August 15, 2024",revenue,by 20%,rise,1,finance,"{'model_name': 'llama-3.3-70b-versatile', 'api_name': 'GROQ_CLOUD', 'batch_id': 0, 'temperature': 0.8, 'top_p': 0.8, 'generation_date': '2026-March-21', 'template_number': 2, 'template_text': 'template 2: On <p_d>, <p_s> speculates the <p_a> at <p_t> will likely <p_sl> by <p_m>%.'}"
2,"American Heart Association forecasts that the heart disease prevalence at rural communities potentially decrease in August 15, 2027",American Heart Association,rural communities,"August 15, 2027",heart disease prevalence,,decrease,1,health,"{'model_name': 'llama-3.3-70b-versatile', 'api_name': 'GROQ_CLOUD', 'batch_id': 0, 'temperature': 0.8, 'top_p': 0.8, 'generation_date': '2026-March-21', 'template_number': 1, 'template_text': 'template 1: <p_s> forecasts that the <p_a> at <p_t> potentially <p_sl> in <p_d>.'}"
3,"On Q2 of 2028, Dr. Smith from Mayo Clinic speculates the average patient wait time at urban hospitals will likely drop by 25%",Dr. Smith from Mayo Clinic,urban hospitals,Q2 of 2028,average patient wait time,by 25%,drop,1,health,"{'model_name': 'llama-3.3-70b-versatile', 'api_name': 'GROQ_CLOUD', 'batch_id': 0, 'temperature': 0.8, 'top_p': 0.8, 'generation_date': '2026-March-21', 'template_number': 2, 'template_text': 'template 2: On <p_d>, <p_s> speculates the <p_a> at <p_t> will likely <p_sl> by <p_m>%.'}"
4,"Senior Economist at Goldman Sachs forecasts that the economic growth at European markets potentially increase in August 15, 2025.",Senior Economist at Goldman Sachs,European markets,"August 15, 2025",economic growth,,increase,1,policy,"{'model_name': 'llama-3.3-70b-versatile', 'api_name': 'GROQ_CLOUD', 'batch_id': 0, 'temperature': 0.8, 'top_p': 0.8, 'generation_date': '2026-March-21', 'template_number': 1, 'template_text': 'template 1: <p_s> forecasts that the <p_a> at <p_t> potentially <p_sl> in <p_d>.'}"
5,"On 2027-09-20, Policy Analyst speculates the job creation at tech industry will likely rise by 10%.",Policy Analyst,tech industry,2027-09-20,job creation,by 10%,rise,1,policy,"{'model_name': 'llama-3.3-70b-versatile', 'api_name': 'GROQ_CLOUD', 'batch_id': 0, 'temperature': 0.8, 'top_p': 0.8, 'generation_date': '2026-March-21', 'template_number': 2, 'template_text': 'template 2: On <p_d>, <p_s> speculates the <p_a> at <p_t> will likely <p_sl> by <p_m>%.'}"
6,"National Weather Service forecasts that the humidity at New York City potentially decrease in August 15, 2024",National Weather Service,New York City,"August 15, 2024",humidity,,decrease,1,weather,"{'model_name': 'llama-3.3-70b-versatile', 'api_name': 'GROQ_CLOUD', 'batch_id': 0, 'temperature': 0.8, 'top_p': 0.8, 'generation_date': '2026-March-21', 'template_number': 1, 'template_text': 'template 1: <p_s> forecasts that the <p_a> at <p_t> potentially <p_sl> in <p_d>.'}"
7,"On 2024-09-20, AccuWeather speculates that the temperature in Chicago will likely rise by 10 degrees",AccuWeather,Chicago,2024-09-20,temperature,by 10 degrees,rise,1,weather,"{'model_name': 'llama-3.3-70b-versatile', 'api_name': 'GROQ_CLOUD', 'batch_id': 0, 'temperature': 0.8, 'top_p': 0.8, 'generation_date': '2026-March-21', 'template_number': 2, 'template_text': 'template 2: On <p_d>, <p_s> speculates the <p_a> at <p_t> will likely <p_sl> by <p_m>%.'}"
8,"Sport Reporter Tom Harris forecasts that 